# Tutorial: PM-02 EM for a Gaussian Mixture Model

Audience:
- Learners studying latent-variable models and the expectation-maximization algorithm.

Prerequisites:
- Multivariate Gaussian basics, Bayes' rule, and the ELBO view of EM.

Learning goals:
- Implement EM for a Gaussian mixture model from scratch.
- Track the observed-data log-likelihood across iterations.
- Visualize soft assignments and fitted component means on synthetic data.


## Outline

1. Generate a two-dimensional mixture dataset.
2. Implement Gaussian log densities and EM updates.
3. Fit a three-component GMM and monitor likelihood improvement.
4. Visualize the learned responsibilities and parameters.


In [ ]:
from __future__ import annotations

import matplotlib
matplotlib.use("Agg")

import matplotlib.pyplot as plt
import numpy as np

SEED = 17
rng = np.random.default_rng(SEED)
SEED


## Step 1 - Generate a synthetic mixture

The data come from three Gaussian components in two dimensions. Because the latent labels are hidden during fitting, EM must infer soft component memberships through responsibilities.


In [ ]:
def make_gmm_dataset(seed: int = SEED):
    local_rng = np.random.default_rng(seed)
    means = np.array([[-2.5, 0.0], [0.5, 2.6], [2.8, -1.2]])
    covs = np.array([
        [[0.60, 0.18], [0.18, 0.40]],
        [[0.55, -0.10], [-0.10, 0.45]],
        [[0.70, 0.22], [0.22, 0.55]],
    ])
    weights = np.array([0.32, 0.36, 0.32])
    counts = local_rng.multinomial(450, weights)
    parts = []
    labels = []
    for idx, (count, mean, cov) in enumerate(zip(counts, means, covs)):
        parts.append(local_rng.multivariate_normal(mean, cov, size=count))
        labels.append(np.full(count, idx, dtype=int))
    X = np.vstack(parts)
    y = np.concatenate(labels)
    order = local_rng.permutation(len(X))
    return X[order], y[order]


X, y_true = make_gmm_dataset()

fig, ax = plt.subplots(figsize=(6.2, 4.6))
scatter = ax.scatter(X[:, 0], X[:, 1], c=y_true, cmap="viridis", s=18, alpha=0.75)
ax.set_title("Synthetic Gaussian mixture sample")
ax.set_xlabel("x1")
ax.set_ylabel("x2")
fig.colorbar(scatter, ax=ax, fraction=0.046, label="true component")
fig.tight_layout()
plt.close(fig)

{"shape": X.shape, "true_component_counts": np.bincount(y_true).tolist()}


## Step 2 - Implement EM ingredients

The E-step needs posterior responsibilities, so we compute Gaussian log densities and normalize in log space. The M-step then uses those responsibilities as soft counts.


In [ ]:
def logsumexp(a: np.ndarray, axis: int = 1, keepdims: bool = False) -> np.ndarray:
    a_max = np.max(a, axis=axis, keepdims=True)
    stabilized = a - a_max
    result = a_max + np.log(np.sum(np.exp(stabilized), axis=axis, keepdims=True))
    if keepdims:
        return result
    return np.squeeze(result, axis=axis)


def gaussian_logpdf(X: np.ndarray, mean: np.ndarray, cov: np.ndarray) -> np.ndarray:
    d = X.shape[1]
    centered = X - mean
    sign, logdet = np.linalg.slogdet(cov)
    if sign <= 0:
        raise ValueError("Covariance must be positive definite.")
    solved = np.linalg.solve(cov, centered.T).T
    quad = np.sum(centered * solved, axis=1)
    return -0.5 * (d * np.log(2.0 * np.pi) + logdet + quad)


class GaussianMixtureEM:
    def __init__(self, n_components: int, reg_covar: float = 1e-4, max_iter: int = 60, tol: float = 1e-4, seed: int = SEED):
        self.n_components = n_components
        self.reg_covar = reg_covar
        self.max_iter = max_iter
        self.tol = tol
        self.seed = seed

    def _initialize(self, X: np.ndarray) -> None:
        local_rng = np.random.default_rng(self.seed)
        indices = local_rng.choice(len(X), size=self.n_components, replace=False)
        self.means_ = X[indices].copy()
        base_cov = np.cov(X.T) + self.reg_covar * np.eye(X.shape[1])
        self.covariances_ = np.repeat(base_cov[None, :, :], self.n_components, axis=0)
        self.weights_ = np.full(self.n_components, 1.0 / self.n_components)
        self.log_likelihood_history_ = []

    def _e_step(self, X: np.ndarray) -> tuple[np.ndarray, float]:
        log_joint = []
        for k in range(self.n_components):
            log_joint.append(np.log(self.weights_[k]) + gaussian_logpdf(X, self.means_[k], self.covariances_[k]))
        log_joint = np.column_stack(log_joint)
        log_norm = logsumexp(log_joint, axis=1, keepdims=True)
        log_resp = log_joint - log_norm
        return np.exp(log_resp), float(np.sum(log_norm))

    def _m_step(self, X: np.ndarray, responsibilities: np.ndarray) -> None:
        n_samples, n_features = X.shape
        Nk = responsibilities.sum(axis=0) + 1e-12
        self.weights_ = Nk / n_samples
        self.means_ = (responsibilities.T @ X) / Nk[:, None]
        covariances = []
        for k in range(self.n_components):
            centered = X - self.means_[k]
            weighted = responsibilities[:, [k]] * centered
            cov = (weighted.T @ centered) / Nk[k]
            cov += self.reg_covar * np.eye(n_features)
            covariances.append(cov)
        self.covariances_ = np.stack(covariances)

    def fit(self, X: np.ndarray) -> "GaussianMixtureEM":
        self._initialize(X)
        previous = -np.inf
        for _ in range(self.max_iter):
            responsibilities, log_likelihood = self._e_step(X)
            self._m_step(X, responsibilities)
            self.log_likelihood_history_.append(log_likelihood)
            if abs(log_likelihood - previous) < self.tol:
                break
            previous = log_likelihood
        self.responsibilities_, self.log_likelihood_ = self._e_step(X)
        self.labels_ = np.argmax(self.responsibilities_, axis=1)
        return self


gmm = GaussianMixtureEM(n_components=3, max_iter=80, tol=1e-5, seed=SEED).fit(X)
{"iterations": len(gmm.log_likelihood_history_), "final_log_likelihood": round(gmm.log_likelihood_, 2)}


## Step 3 - Confirm monotonic likelihood improvement

EM should not decrease the observed-data log-likelihood. Tracking the history gives an empirical check of the theory from the derivation note.


In [ ]:
history = np.array(gmm.log_likelihood_history_)
increments = np.diff(history)

fig, ax = plt.subplots(figsize=(6.0, 3.8))
ax.plot(np.arange(1, len(history) + 1), history, marker="o", linewidth=1.8, ms=3)
ax.set_title("Observed-data log-likelihood during EM")
ax.set_xlabel("iteration")
ax.set_ylabel("log-likelihood")
fig.tight_layout()
plt.close(fig)

{"minimum_increment": round(float(increments.min(initial=0.0)), 8), "monotone_non_decreasing": bool(np.all(increments >= -1e-8))}


## Step 4 - Visualize soft assignments and learned parameters

The color intensity below comes from the fitted responsibilities rather than from hard labels alone. This makes the difference between probabilistic clustering and K-means visible.


In [ ]:
confidence = gmm.responsibilities_.max(axis=1)

fig, ax = plt.subplots(figsize=(6.2, 4.8))
scatter = ax.scatter(X[:, 0], X[:, 1], c=gmm.labels_, cmap="viridis", s=20, alpha=0.25 + 0.75 * confidence)
ax.scatter(gmm.means_[:, 0], gmm.means_[:, 1], marker="X", s=220, c="red", edgecolor="black", linewidth=1.0, label="learned means")
ax.set_title("EM-fitted GMM with soft-assignment confidence")
ax.set_xlabel("x1")
ax.set_ylabel("x2")
ax.legend(loc="upper left")
fig.colorbar(scatter, ax=ax, fraction=0.046, label="assigned component")
fig.tight_layout()
plt.close(fig)

{
    "weights": np.round(gmm.weights_, 3).tolist(),
    "means": np.round(gmm.means_, 3).tolist(),
}


## Summary

This notebook fit a Gaussian mixture model with EM by alternating between posterior responsibilities and weighted parameter updates. The log-likelihood rose monotonically, and the final visualization showed that EM produces soft cluster assignments together with estimated mixture weights, means, and covariances.
